In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path

# =========================================================
# CONFIGURACIÓN
# =========================================================

outputXSize = 800
latRate = 1/4 
nTilesInVArea = 4.5
latMargin = outputXSize * latRate / 2
outputYSize = int((outputXSize - (2 * latMargin)) * nTilesInVArea)
WIDTH, HEIGHT = outputXSize, outputYSize

for i in range(60):
    # Cargar imagen
    input_path = f"real/img_{i}.jpg"

    img = cv2.imread(input_path)
    if img is None:
        print("No se pudo cargar la imagen.")
        continue

    Base_W = 3280 
    Base_H = 2464
    h, w = img.shape[:2]

    scale_x = w / Base_W
    scale_y = h / Base_H

    # Los 4 puntos del trapecio del suelo (Initial)
    pts_src = np.array([
        [322,  2463],  
        [2956, 2463],  
        [1886,  872],  
        [1394,  872]   
    ], dtype=np.float32)

    pts_src[:, 0] *= scale_x  
    pts_src[:, 1] *= scale_y  

    pts_dst = np.array([
        [latMargin, HEIGHT],
        [WIDTH - latMargin, HEIGHT],
        [WIDTH - latMargin, 0],
        [latMargin, 0]
    ], dtype=np.float32)

    # 1. H_INITIAL: Calcula el movimiento original
    H = cv2.getPerspectiveTransform(pts_src, pts_dst)
    
    processed_orig = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # lower_brown = np.array([10, 40, 20])
    # upper_brown = np.array([30, 255, 255])
    lower_brown = np.array([0, 12, 87])
    upper_brown = np.array([67, 137, 200])
    mask_orig = cv2.inRange(processed_orig, lower_brown, upper_brown)
    
    mask_temp = cv2.warpPerspective(mask_orig, H, (WIDTH, HEIGHT), borderValue=0)
    kernel = np.ones((9,9), np.uint8)
    mask_temp = cv2.morphologyEx(mask_temp, cv2.MORPH_OPEN, kernel)
    mask_temp = cv2.morphologyEx(mask_temp, cv2.MORPH_CLOSE, kernel)

    # --- CALCULAR ROTACIÓN ---
    bottom_30 = int(HEIGHT * 0.7)
    mask_bottom = mask_temp[bottom_30:, :]
    edges = cv2.Canny(mask_bottom, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40, minLineLength=40, maxLineGap=10)
    
    angles = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if y2 > y1:
                x1, y1, x2, y2 = x2, y2, x1, y1
                
            dx = x2 - x1
            dy = y1 - y2
            
            if dx != 0 or dy != 0:
                angle_from_horizontal = np.degrees(np.arctan2(dy, dx))
                tilt = 90.0 - angle_from_horizontal 
                if -5 < tilt < 5:
                    angles.append(tilt)

    best_tilt = 0
    if angles:
        best_tilt = np.median(angles)
        
    # --- CALCULAR OFFSET ---
    bottom_15 = int(HEIGHT * 0.85)
    mask_scan = mask_temp[bottom_15:, :]
    col_sums = np.mean(mask_scan, axis=0)
    is_wall = col_sums > (255 * 0.3)
    mid_x = WIDTH // 2
    
    left_wall_found = False
    left_wall_x = 0
    for x in range(mid_x, -1, -1):
        if is_wall[x]:
            left_wall_x = x
            left_wall_found = True
            break
            
    right_wall_found = False
    right_wall_x = WIDTH - 1
    for x in range(mid_x, WIDTH):
        if is_wall[x]:
            right_wall_x = x
            right_wall_found = True
            break

    shift_x = 0
    expected_left = int(latMargin)
    expected_right = int(WIDTH - latMargin)
    
    if left_wall_found and right_wall_found:
        current_center = (left_wall_x + right_wall_x) // 2
        shift_x = mid_x - current_center
    elif left_wall_found and not right_wall_found:
        shift_x = expected_left - left_wall_x
    elif right_wall_found and not left_wall_found:
        shift_x = expected_right - right_wall_x
    
    # Convertimos los movimientos Affine a matrices 3x3
    pivot_center = (WIDTH // 2, HEIGHT) 
    M_rot = cv2.getRotationMatrix2D(pivot_center, best_tilt, 1.0)
    M_rot_3x3 = np.vstack([M_rot, [0, 0, 1]]) # Añadir la 3ra dimensión
    
    M_trans = np.float32([[1, 0, shift_x], [0, 1, 0]])
    M_trans_3x3 = np.vstack([M_trans, [0, 0, 1]])
    
    # H_FINAL = Translación * Rotación * H_inicial
    H_final = M_trans_3x3 @ M_rot_3x3 @ H
    
    os.makedirs("imagenesCenitales", exist_ok=True)
    nombre = os.path.splitext(os.path.basename(input_path))[0]
    extension = os.path.splitext(input_path)[1]
    
    top_down_final = cv2.warpPerspective(img, H_final, (WIDTH, HEIGHT))
    
    mask_final = cv2.warpPerspective(mask_orig, H_final, (WIDTH, HEIGHT), borderValue=0)
    
    mask_final = cv2.morphologyEx(mask_final, cv2.MORPH_OPEN, kernel)
    mask_final = cv2.morphologyEx(mask_final, cv2.MORPH_CLOSE, kernel)

    output_path_color = f"imagenesCenitales/{nombre}_cenital{extension}"
    output_path_bw = f"imagenesCenitales/{nombre}_cenitalBW{extension}"

    cv2.imwrite(output_path_color, top_down_final)
    cv2.imwrite(output_path_bw, mask_final)
    print("")

In [2]:
# =========================================================
# VISUALIZACIÓN
# =========================================================

# Dibujar puntos usados
img_check = img.copy()

for i, pt in enumerate(pts_src):
    cv2.circle(
        img_check,
        tuple(pt.astype(int)),
        7,
        (0, 255, 0),
        -1
    )

    cv2.putText(
        img_check,
        str(i + 1),
        tuple(pt.astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

# Dibujar trapecio
cv2.polylines(
    img_check,
    [pts_src.astype(np.int32)],
    True,
    (255, 0, 0),
    2
)

#cv2.imshow("Puntos Usados Homografia", img_check)
#cv2.imshow("Vista Cenital", top_down)
print()

In [ ]:
import cv2
import numpy as np
import os

def resize_frame(frame, target_size=(820, 616)):
    """
    Resize image for camera processing
    """
    if frame is None:
        return None
        
    return cv2.resize(frame, target_size)


def generate_mapping_sources(input_path, output_dir="testOutput"):
    """
    Saves the b&w homography + reduced resolution version of the camera image
    """
    # =========================================================
    # CONFIGURATION
    # =========================================================
    outputXSize = 800
    latRate = 1/4 
    nTilesInVArea = 4.5
    latMargin = outputXSize * latRate / 2
    outputYSize = int((outputXSize - (2 * latMargin)) * nTilesInVArea)
    WIDTH, HEIGHT = outputXSize, outputYSize
    
    # Load image
    img = cv2.imread(input_path)
    if img is None:
        print(f"Error: Could not load image at {input_path}")
        return None

    Base_W = 3280 
    Base_H = 2464
    h, w = img.shape[:2]

    scale_x = w / Base_W
    scale_y = h / Base_H

    # =========================================================
    # 1. INITIAL HOMOGRAPHY
    # =========================================================
    pts_src = np.array([
        [322,  2463],  
        [2956, 2463],  
        [1886,  872],  
        [1394,  872]   
    ], dtype=np.float32)

    pts_src[:, 0] *= scale_x  
    pts_src[:, 1] *= scale_y  

    pts_dst = np.array([
        [latMargin, HEIGHT],
        [WIDTH - latMargin, HEIGHT],
        [WIDTH - latMargin, 0],
        [latMargin, 0]
    ], dtype=np.float32)

    H = cv2.getPerspectiveTransform(pts_src, pts_dst)
    
    # =========================================================
    # 2. COLOR MASKING & INITIAL WARP
    # =========================================================
    processed_orig = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # lower_brown = np.array([10, 40, 20])
    # upper_brown = np.array([30, 255, 255])
    lower_brown = np.array([0, 12, 87])
    upper_brown = np.array([67, 137, 200])
    mask_orig = cv2.inRange(processed_orig, lower_brown, upper_brown)
    
    mask_temp = cv2.warpPerspective(mask_orig, H, (WIDTH, HEIGHT), borderValue=0)
    kernel = np.ones((9,9), np.uint8)
    mask_temp = cv2.morphologyEx(mask_temp, cv2.MORPH_OPEN, kernel)
    mask_temp = cv2.morphologyEx(mask_temp, cv2.MORPH_CLOSE, kernel)

    # =========================================================
    # 3. CALCULATE ROTATION
    # =========================================================
    bottom_30 = int(HEIGHT * 0.7)
    mask_bottom = mask_temp[bottom_30:, :]
    edges = cv2.Canny(mask_bottom, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40, minLineLength=40, maxLineGap=10)
    
    angles = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if y2 > y1:
                x1, y1, x2, y2 = x2, y2, x1, y1
                
            dx = x2 - x1
            dy = y1 - y2
            
            if dx != 0 or dy != 0:
                angle_from_horizontal = np.degrees(np.arctan2(dy, dx))
                tilt = 90.0 - angle_from_horizontal 
                if -5 < tilt < 5:
                    angles.append(tilt)

    best_tilt = np.median(angles) if angles else 0
        
    # =========================================================
    # 4. CALCULATE OFFSET
    # =========================================================
    bottom_15 = int(HEIGHT * 0.85)
    mask_scan = mask_temp[bottom_15:, :]
    col_sums = np.mean(mask_scan, axis=0)
    is_wall = col_sums > (255 * 0.3)
    mid_x = WIDTH // 2
    
    left_wall_found = False
    left_wall_x = 0
    for x in range(mid_x, -1, -1):
        if is_wall[x]:
            left_wall_x = x
            left_wall_found = True
            break
            
    right_wall_found = False
    right_wall_x = WIDTH - 1
    for x in range(mid_x, WIDTH):
        if is_wall[x]:
            right_wall_x = x
            right_wall_found = True
            break

    shift_x = 0
    expected_left = int(latMargin)
    expected_right = int(WIDTH - latMargin)
    
    if left_wall_found and right_wall_found:
        current_center = (left_wall_x + right_wall_x) // 2
        shift_x = mid_x - current_center
    elif left_wall_found and not right_wall_found:
        shift_x = expected_left - left_wall_x
    elif right_wall_found and not left_wall_found:
        shift_x = expected_right - right_wall_x
    
    # =========================================================
    # 5. FINAL TRANSFORMATION (B&W ONLY)
    # =========================================================
    pivot_center = (WIDTH // 2, HEIGHT) 
    M_rot = cv2.getRotationMatrix2D(pivot_center, best_tilt, 1.0)
    M_rot_3x3 = np.vstack([M_rot, [0, 0, 1]]) 
    
    M_trans = np.float32([[1, 0, shift_x], [0, 1, 0]])
    M_trans_3x3 = np.vstack([M_trans, [0, 0, 1]])
    
    H_final = M_trans_3x3 @ M_rot_3x3 @ H
    
    # Warp ONLY the B&W mask
    mask_final = cv2.warpPerspective(mask_orig, H_final, (WIDTH, HEIGHT), borderValue=0)
    
    mask_final = cv2.morphologyEx(mask_final, cv2.MORPH_OPEN, kernel)
    mask_final = cv2.morphologyEx(mask_final, cv2.MORPH_CLOSE, kernel)

    # =========================================================
    # 6. SAVE RESULTS (B&W Mask + Resized Original)
    # =========================================================
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        nombre = os.path.splitext(os.path.basename(input_path))[0]
        extension = os.path.splitext(input_path)[1]
        
        # 6a. Save the B&W Homography
        output_path_bw = os.path.join(output_dir, f"{nombre}_cenitalBW{extension}")
        cv2.imwrite(output_path_bw, mask_final)
        print(f"Saved: {output_path_bw}")
        
        # 6b. Resize and save the original input image
        img_resized = cv2.resize(img, (820, 616))
        output_path_resized = os.path.join(output_dir, f"{nombre}_resized{extension}")
        cv2.imwrite(output_path_resized, img_resized)
        print(f"Saved: {output_path_resized}")
        
    return mask_final

im = f"real/CuartaCasilla.jpg"
generate_mapping_sources(im)
print()

Saved: testOutput\TerceraCasilla_cenitalBW.jpg
Saved: testOutput\TerceraCasilla_resized.jpg

